# 实践1：简单计算链

函数：**f(x, y) = (xy + x²)²**，取 **x = 2.0，y = 3.0**。

## 1. 拆步骤 + 计算图

把 f 拆成四步简单运算：

- 第①步：a = x·y
- 第②步：b = x·x
- 第③步：c = a + b
- 第④步：f = c·c

计算图（从左到右是前向计算，从右往左是反向求梯度）：

```
x ──┬─► a = x·y ──┐
y ──┘             ├──► c = a + b ──► f = c²
x ─────► b = x² ──┘
```

注意：x 走了两条路，一条经过 a，一条经过 b。

## 反向：链式法则从后往前

| 反向经过 | 局部导数 | 累积梯度 |
|---|---|---|
| f → c | ∂f/∂c = 2c | grad_c = 2c |
| c → a | ∂c/∂a = 1 | grad_a = 2c × 1 |
| c → b | ∂c/∂b = 1 | grad_b = 2c × 1 |
| a → x | ∂a/∂x = y | 经 a 路径：2c·y |
| a → y | ∂a/∂y = x | grad_y = 2c·x |
| b → x | ∂b/∂x = 2x | 经 b 路径：2c·2x |

x 有两条路径，梯度要**相加**：

- **df/dx = 2c·y + 2c·2x = 2c(y + 2x)**
- **df/dy = 2c·x**

代入 x=2、y=3：a=6，b=4，c=10，f=100，所以 df/dc=20，**df/dx = 20×(3+4) = 140**，**df/dy = 20×2 = 40**。

下面先用 NumPy 按这张图手算，再用 PyTorch autograd 验证。


In [1]:
import numpy as np

x = 2.0
y = 3.0

# ---------- 前向：按计算图从左到右 ----------
a = x * y
b = x ** 2
c = a + b
f = c ** 2
print("前向：")
print("  a = x*y =", a)
print("  b = x*x =", b)
print("  c = a+b =", c)
print("  f = c^2 =", f)

# ---------- 反向：按计算图从右往左 ----------
grad_c = 2 * c
grad_a = grad_c * 1.0
grad_b = grad_c * 1.0
grad_y = grad_a * x
grad_x = grad_a * y + grad_b * 2 * x
print("反向：")
print("  df/dc =", grad_c)
print("  df/da =", grad_a)
print("  df/db =", grad_b)
print("  df/dx =", grad_x)
print("  df/dy =", grad_y)


前向：
  a = x*y = 6.0
  b = x*x = 4.0
  c = a+b = 10.0
  f = c^2 = 100.0
反向：
  df/dc = 20.0
  df/da = 20.0
  df/db = 20.0
  df/dx = 140.0
  df/dy = 40.0


In [2]:
import torch

tx = torch.tensor(2.0, requires_grad=True)
ty = torch.tensor(3.0, requires_grad=True)

tf = (tx * ty + tx ** 2) ** 2
tf.backward()

print("autograd：")
print("  f     =", tf.item())
print("  df/dx =", tx.grad.item())
print("  df/dy =", ty.grad.item())

print()
print("对比：")
print("  NumPy    : df/dx =", grad_x, ", df/dy =", grad_y)
print("  autograd : df/dx =", tx.grad.item(), ", df/dy =", ty.grad.item())
same = abs(tx.grad.item() - grad_x) < 1e-9 and abs(ty.grad.item() - grad_y) < 1e-9
print("  完全一致：", same)


autograd：
  f     = 100.0
  df/dx = 140.0
  df/dy = 40.0

对比：
  NumPy    : df/dx = 140.0 , df/dy = 40.0
  autograd : df/dx = 140.0 , df/dy = 40.0
  完全一致： True


## 结论

NumPy 手算与 PyTorch autograd 结果完全一致（df/dx = 140，df/dy = 40，误差为 0）。

autograd 做的事和手算一模一样：

1. 前向计算时自动记录计算图（谁用了谁的值）；
2. 调用 backward() 时从 f 出发沿图反向走，每条边乘上局部导数；
3. 多条路径汇入同一个变量时（比如 x 走 a 和 b 两条路），梯度自动相加。
